# 30 - Reading star clipping: the colour constraint holds, and what it costs

**Purpose.** Explain what notebook `29` found when it followed 8033 stars through the sky-pair
night, and what it means for choosing a setting, for someone deciding what to do next. `29` is
written for someone *checking* it; this one is slow on purpose.

It explains three things: why the peaks come from raw subs and not from registered stacks; why
the gain scales exactly while a longer sub loses about 2%; and what the clipped shares mean when
picking a gain and a sub length.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `star_constants.json` and `star_peaks.csv`, with the constants `29` used beside them.
If anything here disagrees with `results/`, `results/` is right and this notebook is the bug. It is
not the Pareto curve either: section 5 puts a few published numbers through `model.t_max_colour`
to show what they mean, and stops there.

**It assumes** `26_contract3_read.ipynb` and `28_star_mask_read.ipynb` for the night and the SNR
half of the model, and `00_statistics.ipynb` for why a median over many stars is sharp when each
star is not.

**The headline, recomputed in section 1 rather than typed here:** the constraint predicts which
stars clip in all twelve cell-planes.

In [ ]:
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
c = M.Constants.load(RESULTS / "star_constants.json")
peaks = pd.read_csv(RESULTS / "star_peaks.csv")
stack = M.Constants.load(RESULTS / "stack_constants.json")
cold = json.loads((RESULTS / "cold_constants.json").read_text())
mc = json.loads((RESULTS / "model_constants.json").read_text())

CELLS = {"A": "gain 50, 30 s", "B": "gain 50, 120 s", "C": "gain 200, 30 s", "D": "gain 200, 120 s"}
FULL_SCALE = 4095

## 1. The verdict

Cell A - gain 50, 30 s - is where each star's peak was measured. B, C and D are where the model's
prediction was tested. Two bars, both fixed in `29` before any peak was read: the typical star's
measured peak within 10% of its predicted one, and the share of stars that clip inside the range
the prediction gives when every peak is moved by 10% either way.

In [ ]:
scale = pd.DataFrame(c.entry("star_peak_scale")["value"]).T.stack().rename("peak_scale")
se = pd.DataFrame(c.entry("star_peak_scale")["uncertainty"]).T.stack().rename("se")
meas = pd.DataFrame(c.entry("clipped_star_fraction")["value"]).T.stack().rename("clipped")
band = c.entry("clipped_star_fraction_predicted")["uncertainty"]
low = pd.DataFrame(band["low"]).T.stack().rename("low")
high = pd.DataFrame(band["high"]).T.stack().rename("high")
v = pd.concat([scale, se, 100 * meas, 100 * low, 100 * high], axis=1)
v.index.names = ["cell", "plane"]
v["verdict"] = [c.entry("star_clipping_verdict")["value"][f"{a} {b}"] for a, b in v.index]
v.insert(0, "setting", [CELLS[a] for a, _ in v.index])
print(v.round(3).to_string())
print("\n(clipped, low, high in % of the star list)")
print(f"\n{(v.verdict == 'pass').sum()} of {len(v)} cell-planes pass")

**What a pass says.** The constraint is a prediction about where a star's brightest pixel lands.
It passing means two things: the arithmetic in `model.peak_counts` - a star's light times `t`,
divided by `g` - puts the typical star where it really lands, and the count of stars it says will
clip is the count that does.

**What it does not say.** That any one star is predicted to 10%. The next section shows each star
on its own scatters further than that. The constraint is a statement about a *population* of
stars, which is what a clipped share is.

## 2. Why the peaks come from raw subs

Every other number from this night came through PixInsight: register each plane, then stack.
Registration moves each frame onto the reference grid by **resampling** - every output pixel is a
weighted mix of its neighbours. `25` measured what that does to noise: per pixel, a registered
frame reads about 10% quieter than the raw one.

In [ ]:
print("registered / raw single-frame noise (25):", stack.entry("resampling_factor")["value"])

Noise is not the only thing a mix of neighbours flattens. On a Bayer sub-plane a star is only a
few pixels across, so its brightest pixel and the ones beside it differ a lot, and a mix of them
is lower than the brightest. A registered peak is **not the number the camera clipped**, and the
star-colour constraint is about exactly that number.

So `29` registered nothing. It lined each raw sub up with the reference by **whole-pixel shifts**,
which relabel pixels without mixing them - every value it read is a value the camera wrote. One
shift per frame was not enough, because the field turns slowly over the night and the corners
drift by about 2 px; so it used one shift per tile of a 4 x 4 grid, and every star landed within
about a pixel of where the list put it. The peak is the brightest pixel within 1 px.

**The cost is that one star is noisy.** Where a star falls on a pixel moves its peak from sub to
sub, and seeing does too. A median over 36 subs takes most of that out, but not all of it.

In [ ]:
o = peaks[peaks.cell != "A"]
mid = o[(o.ratio > 0) & ~o.clipped & (o.peak_predicted < 0.9 * FULL_SCALE)
        & (o.peak_predicted > 1000)]
q = mid.groupby("cell").ratio.quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()
print("measured / predicted peak, one star at a time, stars predicted 1000-3700 counts:")
print(q.round(3).to_string())

Half of all stars land within about 5-10% of their prediction, and one in five lands more than
about 15% off. But the *middle* of thousands of them is pinned to about half a percent (section 1's
`se`). That is the same arithmetic as everywhere in this project: the spread of one is wide, and the
spread of a median over many is that spread over the square root of how many (`00`).

## 3. The gain is exact; a longer sub loses about 2%

The four cells were chosen so that each change from A moves one thing:

- **A to B** moves only the sub length, 30 s to 120 s, at gain 50. `g` cancels.
- **A to C** moves only the gain, 50 to 200, at 30 s. `t` cancels.
- **A to D** moves both, so it should be roughly B's change times C's.

In [ ]:
# per plane, not pooled: G1 and G2 are the same stars, so four planes are not four draws
v["sigmas_from_1"] = (v.peak_scale - 1) / v.se
by_cell = v.groupby(level="cell").agg(peak_scale=("peak_scale", "mean"),
                                      sigmas_least=("sigmas_from_1", lambda s: s.abs().min()),
                                      sigmas_most=("sigmas_from_1", lambda s: s.abs().max()))
print(by_cell.round(3).to_string())
b, cc, d = by_cell.peak_scale[["B", "C", "D"]]
print(f"\nB x C = {b * cc:.4f} against D = {d:.4f}")

**The gain is exact at a star core.** C lands on its prediction to within its own error.
`g` at gains 50 and 200 was measured on the bench, on flat light, by `15`; a star core is a very
different kind of light, and the ratio between the two gains holds on it anyway. Nothing about
switching gain needs a correction for stars.

**A longer sub loses about 2%, and it is real.** B is about 2% short, three to four standard
errors below 1 in every plane on its own. Only `t` moved, so the extra 90 s did not quite multiply the peak by four. The likely
cause is blur: over two minutes the guiding wanders and the seeing averages over more of its
wobble, so the same light is spread over a slightly wider core. D carries B's loss on top of C's
none, which is what that cause predicts.

**Which way it errs is what matters.** The model assumes the peak grows exactly with `t`, so it
predicts slightly *more* clipping in long subs than happens. That is the safe side: it would tell
you to stop a sub a little early, never to run one that clips. At 2% it changes no decision, which
is why `29` added no term for it. A worse-guided night could make it larger, and then it would.

## 4. Near full scale the check cannot tell

The model's ceiling is not 4095. It is `linear_to_at_least`, the highest level the linearity
sweep *proved* straight on a flat - a few percent lower. The open question was whether a star core
behaves the same: does the response bend just before 4095?

`29` binned the ratio by how close each star's predicted peak came to full scale, and it dips by
1-3% in the top bins. That looks like a bend. It cannot be read as one:

In [ ]:
top = o[(o.peak_predicted >= 0.95 * FULL_SCALE) & (o.peak_predicted < FULL_SCALE)]
print(f"stars predicted at 95-100% of full scale: {len(top)}; "
      f"of those, read at full scale: {int(top.clipped.sum())} "
      f"({100 * top.clipped.mean():.0f}%)")
print("linear_to_at_least:", mc["linear_to_at_least"]["value"])

A star predicted just under 4095 that really lands just over it is *read* as 4095, because that is
as high as the camera goes. Its ratio is capped below where it belongs. In the top bin a good share
of stars hit that cap, so the bin's median is dragged down whether or not anything bends. A bend
before 4095 is **neither shown nor ruled out**.

So the model keeps `linear_to_at_least` as its ceiling, for MISSION's reason: a star stopped a few
percent short loses almost nothing, and a star trusted past what was proved can lose its colour.

## 5. What the clipped shares mean for picking a setting

First the shares themselves, A included. A is the reference, so its share is just what was
measured there.

In [ ]:
share = (100 * peaks.groupby(["cell", "plane"]).clipped.mean()).unstack("plane")
share.insert(0, "setting", [CELLS[x] for x in share.index])
print("% of the 8033 listed stars whose core clips:")
print(share.round(2).to_string())

**Gain 200 at 120 s clips about one listed star in twenty in green**; red clips most and blue
least, in every cell. At gain 50 and 30 s it is about one in five hundred.
These are shares of *this list* - isolated stars in one field - so they are a description of this
night, not a constant of the camera.

What carries to another night is `F_star_peak`: each star's light on its brightest pixel, in
electrons per second, the input `model.t_max_colour` takes. Through it the shares turn into an
answer to the question you actually ask at the mount: **how long can a sub be before the brightest
stars clip?**

In [ ]:
fsp = c.entry("F_star_peak")["value"]
G = {int(k): v["G1"] for k, v in cold["system_gain_cold_per_plane"]["value"].items()}
PED = {int(k): v for k, v in cold["pedestal_cold"]["value"].items()}
CEIL = {int(k): v for k, v in mc["linear_to_at_least"]["value"].items()}
F_SKY = mc["F_sky"]["value"]["G1"]
DARK = mc["dark_current_bound"]["value"]

rows = []
for q in ("q0.9", "q0.99", "q0.999"):
    f = fsp["G1"][q]
    rows.append({"star": f"brightest {100 * (1 - float(q[1:])):g}% of the list", "F_star_peak": f,
                 **{f"gain {g}": M.t_max_colour(f, f_sky=F_SKY, dark=DARK, pedestal=PED[g],
                                                ceiling=CEIL[g], g=G[g]) for g in (50, 200)}})
print("longest sub, s, before that star's core reaches linear_to_at_least, green (G1):")
print(pd.DataFrame(rows).round(1).to_string(index=False))

**This is the star-colour half of the trade, in one table.** At gain 200 the brightest 1% of these
stars reach the ceiling in tens of seconds, and the brightest 0.1% in a few. At gain 50 every row
is about six times longer, because gain 50 spends about six times as many electrons per count. The
SNR half says the opposite: longer subs and the high-gain mode both buy SNR. The Pareto curve is
those two pulling against each other, plane by plane.

**Three cautions before anyone uses these numbers.** `F_star_peak` belongs to NGC 7000, to this
night's seeing and to this focus: a softer night spreads every star and lowers every row. The very
brightest stars are not in the table at all, because they clip even in A and only a lower bound is
known. And the 2% time loss of section 3 is left out, which makes every long sub here slightly too
short - the safe side.

## 6. What is settled, and what is next

**Settled.**
- `model.peak_counts` predicts where star cores land, and which clip, in every cell and plane of
  this night. The star-colour constraint, the half of the model that had never met a star, passes.
- The gain ratio measured on bench flats holds exactly on star cores.
- Both halves of the model have now passed on the same night: the SNR ranking in `27`, the
  clipping here.

**Not settled, named rather than fixed.**
1. **The Pareto curve itself.** Both halves can now be trusted on this night, so they can be put
   together over `t` and gain. That is the next notebook, and it needs its own agreed purpose.
2. **A longer sub loses about 2% of its peak.** Safe-side, below any decision, and with no model
   term. A worse-guided night is where it would grow.
3. **Whether a star core bends before 4095.** The check is capped by the clip itself. The model keeps
   the proved ceiling.
4. **One night, one field.** Every share and every `F_star_peak` here describes NGC 7000 on
   2026-09-05.
5. **Still open from `28`:** blue, gain 50, 30 to 120 s, overshoots its SNR prediction by 4.8
   repeatabilities.